# ⚡ Quick Run Mode: Fast vs Full

This notebook supports a global run mode (EQUIPAY_MODE) for FAST/FULL runs.


In [ ]:
# GLOBAL RUN MODE (inserted)
import os
EQUIPAY_MODE = os.environ.get('EQUIPAY_MODE', 'FAST')  # FAST | FULL
if EQUIPAY_MODE == 'FAST':
    N_SAMPLES = 1000
    N_BOOTSTRAP = 100
    MAX_ITER = 200
    N_ESTIMATORS = 50
    PLOT_INLINE = False
else:
    N_SAMPLES = None
    N_BOOTSTRAP = 1000
    MAX_ITER = 1000
    N_ESTIMATORS = 200
    PLOT_INLINE = True
print(f"EQUIPAY_MODE={EQUIPAY_MODE}; N_SAMPLES={N_SAMPLES}; N_BOOTSTRAP={N_BOOTSTRAP}")


In [ ]:
# RUN-MODE UTILITIES (safe)
import os
EQUIPAY_MODE = globals().get('EQUIPAY_MODE', os.environ.get('EQUIPAY_MODE','FAST'))

# Better defaults: 10K for FAST mode, 100K for FULL mode  
if EQUIPAY_MODE == 'FAST':
    N_SAMPLES = globals().get('N_SAMPLES', 10000)
else:
    N_SAMPLES = globals().get('N_SAMPLES', 100000)

N_BOOTSTRAP = globals().get('N_BOOTSTRAP', 100)
MAX_ITER = globals().get('MAX_ITER', 200)
N_ESTIMATORS = globals().get('N_ESTIMATORS', 50)

def enforce_fast_sample(df, n=None, seed=42):
    if EQUIPAY_MODE == 'FAST' and n is not None:
        return df.sample(n=min(len(df), n), random_state=seed)
    return df

# Conservative default: in FAST mode we skip small groups to avoid heavy per-group loops.
# In FULL mode we allow small groups to be processed for comprehensive analysis.
MIN_GROUP_N = 1000 if EQUIPAY_MODE == 'FAST' else 30
print(f"EQUIPAY_MODE={EQUIPAY_MODE}; MIN_GROUP_N={MIN_GROUP_N}")

from src.notebook_utils import ensure_store_and_sample, get_sample_from_store, safe_weight_col
store, df_sample = ensure_store_and_sample(sample_limit=N_SAMPLES)
weight_col = safe_weight_col(df_sample)

# Data loading feedback
print(f"✓ Loaded {len(df_sample):,} records")
print(f"✓ Dataset has {len(df_sample.columns)} columns")
if 'SURVYEAR' in df_sample.columns:
    year_range = f"{df_sample['SURVYEAR'].min():.0f}-{df_sample['SURVYEAR'].max():.0f}"
    print(f"✓ Year range: {year_range}")

In [ ]:
# ============================================================================
# SETUP AND IMPORTS
# ============================================================================

import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path
import json
from datetime import datetime

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

from data_store import EquiPayDataStore, Agg, Func

# Publication style settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (8, 6),
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'font.family': 'serif',
    'text.usetex': False  # Set True if LaTeX is available
})

# Output directories
output_dir = PROJECT_ROOT / 'reports' / 'publication'
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / 'figures').mkdir(exist_ok=True)
(output_dir / 'tables').mkdir(exist_ok=True)

print("✅ Setup complete (6-Layer Data Store Architecture)")
print(f"   Output directory: {output_dir}")

In [ ]:
# ============================================================================
# SETUP: Import Libraries and Configure Environment
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Statistical models
import statsmodels.api as sm
from statsmodels.regression.linear_model import WLS

# Project imports
from src.constants import COLS, normalize_column_names, PROV_LABELS, EDUC_LABELS, NOC_10_LABELS
from src.gap_analysis import (
    OaxacaBlinderDecomposition,
    QuantileGapAnalyzer,
    IntersectionalAnalyzer,
)
from src.gap_analysis.core import weighted_mean, weighted_quantile

# Publication-quality figure settings (APA style)
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.8,
    'grid.linewidth': 0.5,
    'lines.linewidth': 1.5,
})

# Output directories
FIGURES_DIR = Path('../reports/figures/publication')
TABLES_DIR = Path('../reports/tables')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Libraries loaded successfully")
print(f"✓ Figures will be saved to: {FIGURES_DIR}")
print(f"✓ Tables will be saved to: {TABLES_DIR}")

In [ ]:
# ============================================================================
# DATA LOADING VIA EquiPayDataStore (6-Layer Architecture)
# ============================================================================

# Initialize data store with new architecture
store = EquiPayDataStore(
    parquet_path=str(PROJECT_ROOT / "data" / "parquet"),
    memory_limit_mb=1000,
    enable_cache=True
)

# Load data with valid wages using new API
try:
    df = store.query().where('HRLYEARN > 0').to_pandas()
except Exception:
    df = get_sample_from_store(query='HRLYEARN > 0', limit=N_SAMPLES)

df = normalize_column_names(df)

# Create analysis variables
df['IS_FEMALE'] = (df['GENDER'] == 2).astype(int)
df['LOG_WAGE'] = np.log(df['REAL_HRLYEARN'].clip(lower=1))
df['GENDER_LABEL'] = df['GENDER'].map({1: 'Male', 2: 'Female'})
df['PROV_LABEL'] = df['PROV'].map(PROV_LABELS)
df['EDUC_LABEL'] = df['EDUC'].map(EDUC_LABELS)
df['NOC_LABEL'] = df['NOC_10'].map(NOC_10_LABELS)

# Get summary stats
total_records = store.count()
years = store.years()

print(f"✓ Loaded {len(df):,} records (6-Layer Architecture)")
print(f"✓ Years: {min(years)} - {max(years)}")

## Table 1: Summary Statistics by Gender

In [ ]:
# ============================================================================
# TABLE 1: SUMMARY STATISTICS BY GENDER
# ============================================================================

print("=" * 70)
print("TABLE 1: Summary Statistics by Gender")
print("=" * 70)

def weighted_stats(series, weights):
    """Calculate weighted mean and SD."""
    mean = np.average(series, weights=weights)
    variance = np.average((series - mean)**2, weights=weights)
    sd = np.sqrt(variance)
    return mean, sd

# Variables to summarize
vars_to_summarize = [
    ('REAL_HRLYEARN', 'Hourly Wage (2010$)'),
    ('AGE_12', 'Age Category'),
    ('EDUC', 'Education Level'),
    ('TENURE', 'Job Tenure'),
]

table1_rows = []

for var, label in vars_to_summarize:
    if var not in df.columns:
        continue
    
    for gender in ['Male', 'Female']:
        subset = df[df['GENDER_LABEL'] == gender]
        valid = subset[var].notna()
        mean, sd = weighted_stats(
            subset.loc[valid, var].values,
            subset.loc[valid, 'FINALWT'].values
        )
        table1_rows.append({
            'Variable': label,
            'Gender': gender,
            'Mean': mean,
            'SD': sd,
            'N': valid.sum()
        })

# Add sample sizes (SQL; faster and central)
sample_df = store.sql("""
SELECT GENDER_LABEL AS Gender,
       'Sample Size (unweighted)' AS Variable,
       COUNT(*) AS Mean,
       NULL AS SD,
       COUNT(*) AS N
FROM lfs_enriched
GROUP BY GENDER_LABEL
UNION ALL
SELECT GENDER_LABEL AS Gender,
       'Population (weighted)' AS Variable,
       SUM(FINALWT) AS Mean,
       NULL AS SD,
       COUNT(*) AS N
FROM lfs_enriched
GROUP BY GENDER_LABEL
""")

# Convert SQL result to the same table1_rows format and append
for _, r in sample_df.iterrows():
    table1_rows.append({
        'Variable': r['Variable'],
        'Gender': r['Gender'],
        'Mean': r['Mean'],
        'SD': np.nan,
        'N': int(r['N'])
    })

table1 = pd.DataFrame(table1_rows)

# Pivot for publication format (small DF; pivot in Python is fine)
table1_pivot = table1.pivot(index='Variable', columns='Gender', values=['Mean', 'SD'])

# Parity check (sanity): compare SQL sample sizes with pandas counts
sample_pd = pd.DataFrame([
    {'Variable': 'Sample Size (unweighted)', 'Gender': g, 'Mean': len(df[df['GENDER_LABEL'] == g]), 'N': len(df[df['GENDER_LABEL'] == g])}
    for g in df['GENDER_LABEL'].unique()
])
merged = sample_df.merge(sample_pd, on=['Variable', 'Gender'], suffixes=('_sql', '_pd'))
if not (merged['Mean_sql'] == merged['Mean_pd']).all():
    raise AssertionError('Sample size parity failed between SQL and pandas')
print('✓ Sample size parity OK')

# Calculate difference
table1_export = pd.DataFrame({
    'Variable': table1_pivot.index,
    'Male Mean': table1_pivot[('Mean', 'Male')].values,
    'Male SD': table1_pivot[('SD', 'Male')].values,
    'Female Mean': table1_pivot[('Mean', 'Female')].values,
    'Female SD': table1_pivot[('SD', 'Female')].values,
})

# Add wage difference
wage_row = table1_export[table1_export['Variable'] == 'Hourly Wage (2010$)']
if len(wage_row) > 0:
    male_wage = wage_row['Male Mean'].values[0]
    female_wage = wage_row['Female Mean'].values[0]
    gap = (female_wage - male_wage) / male_wage * 100
    
    table1_export = pd.concat([table1_export, pd.DataFrame([{
        'Variable': 'Gender Gap (%)',
        'Male Mean': np.nan,
        'Male SD': np.nan,
        'Female Mean': gap,
        'Female SD': np.nan,
    }])], ignore_index=True)

print(table1_export.to_string(index=False))

# Save
table1_export.to_csv(TABLES_DIR / 'table1_summary_statistics.csv', index=False)
print(f"\n✓ Saved to {TABLES_DIR / 'table1_summary_statistics.csv'}")

In [ ]:
# Generate LaTeX version
def to_latex_table(df, caption, label, note=""):
    """Convert DataFrame to LaTeX table."""
    latex = "\\begin{table}[htbp]\n"
    latex += "\\centering\n"
    latex += f"\\caption{{{caption}}}\n"
    latex += f"\\label{{{label}}}\n"
    latex += df.to_latex(index=False, float_format="%.2f", na_rep="--")
    if note:
        latex += f"\n\\vspace{{2mm}}\n\\footnotesize{{\\textit{{Note:}} {note}}}\n"
    latex += "\\end{table}\n"
    return latex

latex1 = to_latex_table(
    table1_export,
    "Summary Statistics by Gender",
    "tab:summary",
    "Statistics are weighted using survey weights (FINALWT). Hourly wages are in 2010 constant dollars."
)

with open(TABLES_DIR / 'table1_summary_statistics.tex', 'w') as f:
    f.write(latex1)

print("LaTeX Preview:")
print(latex1[:500] + "...")

## Table 2: Oaxaca-Blinder Decomposition

In [ ]:
# ============================================================================
# TABLE 2: OAXACA-BLINDER DECOMPOSITION
# ============================================================================

print("=" * 70)
print("TABLE 2: Oaxaca-Blinder Decomposition of the Gender Wage Gap")
print("=" * 70)

# Prepare data
df_ob = df[['LOG_WAGE', 'IS_FEMALE', 'EDUC', 'AGE_12', 'TENURE', 
            'NOC_10', 'PROV', 'FINALWT']].dropna()

# Sample for computation
n_sample = min(100_000, len(df_ob))
df_ob_sample = df_ob.sample(n=n_sample, random_state=42)

# Features
features = ['EDUC', 'AGE_12', 'TENURE', 'NOC_10', 'PROV']
X = df_ob_sample[features]
y = df_ob_sample['LOG_WAGE']
group = df_ob_sample['IS_FEMALE']
weights = df_ob_sample['FINALWT']

# Run decomposition
decomp = OaxacaBlinderDecomposition(reference='pooled')
result = decomp.fit(X, y, group, weights=weights, group_a_value=0, group_b_value=1)

# Create table
table2_rows = [
    {'Component': 'Mean Log Wage (Male)', 'Value': weighted_mean(y[group==0].values, weights[group==0].values), 'SE': np.nan, 'Share (%)': np.nan},
    {'Component': 'Mean Log Wage (Female)', 'Value': weighted_mean(y[group==1].values, weights[group==1].values), 'SE': np.nan, 'Share (%)': np.nan},
    {'Component': 'Total Gap', 'Value': result.mean_difference, 'SE': np.nan, 'Share (%)': 100.0},
    {'Component': '', 'Value': np.nan, 'SE': np.nan, 'Share (%)': np.nan},
    {'Component': 'Explained (Endowments)', 'Value': result.explained, 'SE': result.se_explained, 
     'Share (%)': result.explained / result.mean_difference * 100 if result.mean_difference != 0 else np.nan},
    {'Component': 'Unexplained (Discrimination)', 'Value': result.unexplained, 'SE': result.se_unexplained,
     'Share (%)': result.unexplained / result.mean_difference * 100 if result.mean_difference != 0 else np.nan},
]

# Add detailed decomposition if available
if result.detailed_explained is not None:
    table2_rows.append({'Component': '', 'Value': np.nan, 'SE': np.nan, 'Share (%)': np.nan})
    table2_rows.append({'Component': 'Detailed Explained:', 'Value': np.nan, 'SE': np.nan, 'Share (%)': np.nan})
    for var, val in result.detailed_explained.items():
        table2_rows.append({
            'Component': f'  {var}',
            'Value': val,
            'SE': np.nan,
            'Share (%)': val / result.mean_difference * 100 if result.mean_difference != 0 else np.nan
        })

table2 = pd.DataFrame(table2_rows)
print(table2.to_string(index=False))

# Save
table2.to_csv(TABLES_DIR / 'table2_oaxaca_blinder.csv', index=False)
print(f"\n✓ Saved to {TABLES_DIR / 'table2_oaxaca_blinder.csv'}")

## Figure 1: Gender Wage Gap Trends (2010-2025)

In [ ]:
# ============================================================================
# FIGURE 1: GENDER WAGE GAP TRENDS
# ============================================================================

print("=" * 70)
print("FIGURE 1: Gender Wage Gap Trends (2010-2025)")
print("=" * 70)

# Calculate annual gaps
annual_gaps = []

for year in sorted(df['SURVYEAR'].unique()):
    year_df = df[df['SURVYEAR'] == year]
    
    male = year_df[year_df['GENDER'] == 1]
    female = year_df[year_df['GENDER'] == 2]
    
    male_wage = weighted_mean(male['REAL_HRLYEARN'].values, male['FINALWT'].values)
    female_wage = weighted_mean(female['REAL_HRLYEARN'].values, female['FINALWT'].values)
    
    gap = (female_wage - male_wage) / male_wage * 100
    
    # Bootstrap SE
    np.random.seed(42)
    boot_gaps = []
    for _ in range(200):
        m_idx = np.random.choice(len(male), len(male), replace=True)
        f_idx = np.random.choice(len(female), len(female), replace=True)
        m_w = weighted_mean(male['REAL_HRLYEARN'].values[m_idx], male['FINALWT'].values[m_idx])
        f_w = weighted_mean(female['REAL_HRLYEARN'].values[f_idx], female['FINALWT'].values[f_idx])
        boot_gaps.append((f_w - m_w) / m_w * 100)
    
    se = np.std(boot_gaps)
    
    annual_gaps.append({
        'Year': year,
        'Gap (%)': gap,
        'SE': se,
        'CI_lower': gap - 1.96 * se,
        'CI_upper': gap + 1.96 * se
    })

annual_df = pd.DataFrame(annual_gaps)

In [ ]:
# Create publication-quality figure
fig, ax = plt.subplots(figsize=(8, 5))

# Plot with confidence interval
ax.fill_between(annual_df['Year'], annual_df['CI_lower'], annual_df['CI_upper'],
                alpha=0.2, color='#1f77b4', label='95% CI')
ax.plot(annual_df['Year'], annual_df['Gap (%)'], 'o-', color='#1f77b4',
        linewidth=2, markersize=6, label='Gender Gap')

# Reference line
ax.axhline(y=0, color='black', linewidth=0.8)

# Add policy markers
ax.axvline(x=2020, color='red', linestyle='--', alpha=0.5, linewidth=1)
ax.axvline(x=2021, color='green', linestyle='--', alpha=0.5, linewidth=1)

# Annotations
ax.annotate('COVID-19', xy=(2020, ax.get_ylim()[1] * 0.95),
            fontsize=8, color='red', ha='center')
ax.annotate('Federal\nPay Equity\nAct', xy=(2021, ax.get_ylim()[1] * 0.85),
            fontsize=8, color='green', ha='center')

# Labels
ax.set_xlabel('Year', fontsize=10)
ax.set_ylabel('Gender Wage Gap (%)', fontsize=10)
ax.set_title('Figure 1: Gender Wage Gap Trends in Canada, 2010-2025',
             fontsize=11, fontweight='bold')

# Legend
ax.legend(loc='lower right', frameon=True, edgecolor='gray')

# Clean up
ax.set_xlim(2009.5, 2025.5)
ax.set_xticks(range(2010, 2026, 2))

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure1_gap_trends.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'figure1_gap_trends.pdf', bbox_inches='tight')
plt.show()

print(f"\n✓ Saved to {FIGURES_DIR / 'figure1_gap_trends.png'}")

## Figure 2: Glass Ceiling Visualization

In [ ]:
# ============================================================================
# FIGURE 2: GLASS CEILING VISUALIZATION
# ============================================================================

print("=" * 70)
print("FIGURE 2: Gender Gap Across the Wage Distribution")
print("=" * 70)

# Calculate gaps at quantiles
quantiles = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]

male_df = df[df['GENDER'] == 1]
female_df = df[df['GENDER'] == 2]

quantile_gaps = []
for q in quantiles:
    male_q = weighted_quantile(male_df['REAL_HRLYEARN'].values, male_df['FINALWT'].values, q)
    female_q = weighted_quantile(female_df['REAL_HRLYEARN'].values, female_df['FINALWT'].values, q)
    gap = (female_q - male_q) / male_q * 100
    quantile_gaps.append({'Quantile': q, 'Gap (%)': gap})

qg_df = pd.DataFrame(quantile_gaps)

# Create figure
fig, ax = plt.subplots(figsize=(8, 5))

# Area fill
ax.fill_between(qg_df['Quantile'], 0, qg_df['Gap (%)'],
                alpha=0.3, color='#d62728')
ax.plot(qg_df['Quantile'], qg_df['Gap (%)'], 'o-', color='#d62728',
        linewidth=2, markersize=8)

# Reference
ax.axhline(y=0, color='black', linewidth=0.8)
median_gap = qg_df[qg_df['Quantile'] == 0.50]['Gap (%)'].values[0]
ax.axhline(y=median_gap, color='gray', linestyle='--', alpha=0.5,
           label=f'Median gap: {median_gap:.1f}%')

# Annotations
ax.annotate('Sticky Floor\n(low earners)', xy=(0.10, qg_df[qg_df['Quantile']==0.10]['Gap (%)'].values[0]),
            xytext=(0.20, qg_df[qg_df['Quantile']==0.10]['Gap (%)'].values[0] + 3),
            fontsize=9, arrowprops=dict(arrowstyle='->', color='gray'))

ax.annotate('Glass Ceiling\n(high earners)', xy=(0.90, qg_df[qg_df['Quantile']==0.90]['Gap (%)'].values[0]),
            xytext=(0.75, qg_df[qg_df['Quantile']==0.90]['Gap (%)'].values[0] - 4),
            fontsize=9, arrowprops=dict(arrowstyle='->', color='gray'))

# Labels
ax.set_xlabel('Wage Quantile (τ)', fontsize=10)
ax.set_ylabel('Gender Wage Gap (%)', fontsize=10)
ax.set_title('Figure 2: Gender Gap Across the Wage Distribution\n(Glass Ceiling Analysis)',
             fontsize=11, fontweight='bold')
ax.legend(loc='best')

ax.set_xticks(quantiles)
ax.set_xticklabels([f'{q:.0%}' for q in quantiles])

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure2_glass_ceiling.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'figure2_glass_ceiling.pdf', bbox_inches='tight')
plt.show()

print(f"\n✓ Saved to {FIGURES_DIR / 'figure2_glass_ceiling.png'}")

## Figure 3: Intersectional Heatmap

In [ ]:
# ============================================================================
# FIGURE 3: INTERSECTIONAL HEATMAP
# ============================================================================

print("=" * 70)
print("FIGURE 3: Intersectional Wage Gaps (Gender × Province × Occupation)")
print("=" * 70)

# Calculate gaps by province and gender
gap_matrix = []

major_provs = [35, 24, 59, 48, 46]  # ON, QC, BC, AB, MB
major_occs = [0, 1, 2, 3, 4]  # First 5 NOC categories

for prov in major_provs:
    for occ in major_occs:
        subset = df[(df['PROV'] == prov) & (df['NOC_10'] == occ)]
        if len(subset) < 200:
            continue
        
        male = subset[subset['GENDER'] == 1]
        female = subset[subset['GENDER'] == 2]
        
        if len(male) < 50 or len(female) < 50:
            continue
        
        m_wage = weighted_mean(male['REAL_HRLYEARN'].values, male['FINALWT'].values)
        f_wage = weighted_mean(female['REAL_HRLYEARN'].values, female['FINALWT'].values)
        gap = (f_wage - m_wage) / m_wage * 100
        
        gap_matrix.append({
            'Province': PROV_LABELS.get(prov, str(prov)),
            'Occupation': NOC_10_LABELS.get(occ, str(occ))[:25],
            'Gap (%)': gap
        })

gap_matrix_df = pd.DataFrame(gap_matrix)
pivot = gap_matrix_df.pivot(index='Province', columns='Occupation', values='Gap (%)')

# Create heatmap
fig, ax = plt.subplots(figsize=(12, 6))

sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
            cbar_kws={'label': 'Gender Gap (%)'}, ax=ax,
            linewidths=0.5, linecolor='white')

ax.set_title('Figure 3: Gender Wage Gap by Province and Occupation',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Occupation (NOC)', fontsize=10)
ax.set_ylabel('Province', fontsize=10)

# Rotate x labels
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure3_intersectional_heatmap.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'figure3_intersectional_heatmap.pdf', bbox_inches='tight')
plt.show()

print(f"\n✓ Saved to {FIGURES_DIR / 'figure3_intersectional_heatmap.png'}")

## Table 3: Provincial Gap Comparison

In [ ]:
# ============================================================================
# TABLE 3: PROVINCIAL GAP COMPARISON
# ============================================================================

print("=" * 70)
print("TABLE 3: Gender Wage Gap by Province")
print("=" * 70)

prov_table = []

# Compute point estimates via SQL (fast)
min_n = MIN_GROUP_N
prov_stats = store.sql(f"""
SELECT PROV AS Province,
       SUM(CASE WHEN GENDER = 1 THEN REAL_HRLYEARN * FINALWT END)/NULLIF(SUM(CASE WHEN GENDER = 1 THEN FINALWT END), 0) AS male_wage,
       SUM(CASE WHEN GENDER = 2 THEN REAL_HRLYEARN * FINALWT END)/NULLIF(SUM(CASE WHEN GENDER = 2 THEN FINALWT END), 0) AS female_wage,
       ( (SUM(CASE WHEN GENDER = 2 THEN REAL_HRLYEARN * FINALWT END)/NULLIF(SUM(CASE WHEN GENDER = 2 THEN FINALWT END), 0))
         - (SUM(CASE WHEN GENDER = 1 THEN REAL_HRLYEARN * FINALWT END)/NULLIF(SUM(CASE WHEN GENDER = 1 THEN FINALWT END), 0))
       ) / NULLIF((SUM(CASE WHEN GENDER = 1 THEN REAL_HRLYEARN * FINALWT END)/NULLIF(SUM(CASE WHEN GENDER = 1 THEN FINALWT END), 0)), 0) * 100 AS gap_pct,
       COUNT(*) AS N
FROM lfs_enriched
GROUP BY PROV
HAVING COUNT(*) >= {min_n}
ORDER BY gap_pct ASC
""")

# Bootstrap SE using small per-province samples fetched via SQL
for _, row in prov_stats.iterrows():
    prov = int(row['Province'])
    sample_df = store.sql(f"""
SELECT REAL_HRLYEARN, FINALWT, GENDER
FROM lfs_enriched
WHERE PROV = {prov}
""")
    male = sample_df[sample_df['GENDER'] == 1]
    female = sample_df[sample_df['GENDER'] == 2]

    # If either group is empty or too small, skip bootstrap
    if len(male) == 0 or len(female) == 0:
        se = float('nan')
    else:
        np.random.seed(42)
        boot_gaps = []
        for _ in range(100):
            m_idx = np.random.choice(len(male), len(male), replace=True)
            f_idx = np.random.choice(len(female), len(female), replace=True)
            m_w = weighted_mean(male['REAL_HRLYEARN'].values[m_idx], male['FINALWT'].values[m_idx])
            f_w = weighted_mean(female['REAL_HRLYEARN'].values[f_idx], female['FINALWT'].values[f_idx])
            boot_gaps.append((f_w - m_w) / m_w * 100)
        se = np.std(boot_gaps)

    prov_table.append({
        'Province': PROV_LABELS.get(prov, str(prov)),
        'Male Wage ($)': row['male_wage'],
        'Female Wage ($)': row['female_wage'],
        'Gap (%)': row['gap_pct'],
        'SE': se,
        'N': int(row['N'])
    })

table3 = pd.DataFrame(prov_table).sort_values('Gap (%)')
print(table3.to_string(index=False))

# Save
table3.to_csv(TABLES_DIR / 'table3_provincial_gaps.csv', index=False)
print(f"\n✓ Saved to {TABLES_DIR / 'table3_provincial_gaps.csv'}")

In [ ]:
# Parity check: compare SQL point estimates with a one-off pandas groupby (sanity check)
prov_pd_rows = []
for prov in prov_stats['Province'].unique():
    prov = int(prov)
    prov_df = df[df['PROV'] == prov]
    if len(prov_df) < MIN_GROUP_N:
        continue
    male = prov_df[prov_df['GENDER'] == 1]
    female = prov_df[prov_df['GENDER'] == 2]
    pd_m = weighted_mean(male['REAL_HRLYEARN'].values, male['FINALWT'].values)
    pd_f = weighted_mean(female['REAL_HRLYEARN'].values, female['FINALWT'].values)
    prov_pd_rows.append({'Province': prov, 'pd_male': pd_m, 'pd_female': pd_f})
prov_pd = pd.DataFrame(prov_pd_rows)
merged = prov_stats.merge(prov_pd, left_on='Province', right_on='Province')
# Numeric parity within small tolerance
if not (np.allclose(merged['male_wage'].astype(float), merged['pd_male'].astype(float), rtol=1e-6, atol=1e-6) and np.allclose(merged['female_wage'].astype(float), merged['pd_female'].astype(float), rtol=1e-6, atol=1e-6)):
    raise AssertionError('Provincial wage point estimate parity failed between SQL and pandas')
print('✓ Provincial point estimate parity OK')

## Summary and Export

In [ ]:
# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("=" * 70)
print("PUBLICATION OUTPUTS GENERATED")
print("=" * 70)

# List all generated files
tables_generated = list(TABLES_DIR.glob('*'))
figures_generated = list(FIGURES_DIR.glob('*'))

print("\nTABLES:")
for f in sorted(tables_generated):
    print(f"  ✓ {f.name}")

print("\nFIGURES:")
for f in sorted(figures_generated):
    print(f"  ✓ {f.name}")

print(f"\n" + "=" * 70)
print("KEY FINDINGS FOR ABSTRACT")
print("=" * 70)

# Calculate key statistics
overall_gap = annual_df['Gap (%)'].mean()
latest_gap = annual_df.iloc[-1]['Gap (%)']
earliest_gap = annual_df.iloc[0]['Gap (%)']
gap_change = latest_gap - earliest_gap

print(f"""
1. OVERALL GENDER GAP:
   • Average gap (2010-2025): {overall_gap:.1f}%
   • Gap in 2010: {earliest_gap:.1f}%
   • Gap in latest year: {latest_gap:.1f}%
   • Change: {gap_change:+.1f} percentage points

2. DECOMPOSITION:
   • Total log wage gap: {result.mean_difference:.4f}
   • Explained by characteristics: {result.explained:.4f} ({result.explained/result.mean_difference*100:.1f}%)
   • Unexplained (discrimination): {result.unexplained:.4f} ({result.unexplained/result.mean_difference*100:.1f}%)

3. GLASS CEILING:
   • Gap at 10th percentile: {qg_df[qg_df['Quantile']==0.10]['Gap (%)'].values[0]:.1f}%
   • Gap at 50th percentile: {qg_df[qg_df['Quantile']==0.50]['Gap (%)'].values[0]:.1f}%
   • Gap at 90th percentile: {qg_df[qg_df['Quantile']==0.90]['Gap (%)'].values[0]:.1f}%

4. PROVINCIAL VARIATION:
   • Smallest gap: {table3.iloc[0]['Province']} ({table3.iloc[0]['Gap (%)']:.1f}%)
   • Largest gap: {table3.iloc[-1]['Province']} ({table3.iloc[-1]['Gap (%)']:.1f}%)

5. DATA:
   • N = {len(df):,} observations
   • Period: {df['SURVYEAR'].min()}-{df['SURVYEAR'].max()}
   • Source: Statistics Canada LFS PUMF
""")

In [ ]:
# Generate README for outputs
readme_content = f"""
# EquiPay Canada: Publication-Ready Outputs

Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}

## Tables

| File | Description |
|------|-------------|
| table1_summary_statistics.csv | Summary statistics by gender |
| table2_oaxaca_blinder.csv | Oaxaca-Blinder decomposition |
| table3_provincial_gaps.csv | Gender gap by province |

## Figures

| File | Description |
|------|-------------|
| figure1_gap_trends.png | Gender wage gap 2010-2025 |
| figure2_glass_ceiling.png | Gap across wage distribution |
| figure3_intersectional_heatmap.png | Gap by province × occupation |

## Data Source

Statistics Canada Labour Force Survey (LFS) Public Use Microdata File (PUMF)
Catalogue: 71M0001X
Period: 2010-2025

## Citation

EquiPay Canada Research Team. ({pd.Timestamp.now().year}). Gender Wage Gap Analysis 
Using Canadian Labour Force Survey Data. [Data analysis and visualization].
"""

with open(TABLES_DIR.parent / 'README.md', 'w') as f:
    f.write(readme_content)

print("\n✓ README generated")
print("\n" + "=" * 70)
print("ALL PUBLICATION OUTPUTS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================================
# LOAD DATA
# ============================================================================

print("Loading LFS data...")

store = LFSDataStore()

df = store.query("""
    SELECT 
        YEAR, SURVMNTH, PROV,
        SEX, AGE, EDUC, IMMIG,
        NOC_10, NAICS_21, COWMAIN, UNION,
        HRLYEARN, FINALWT,
        CASE WHEN SEX = 2 THEN 1 ELSE 0 END AS IS_FEMALE
    FROM lfs_data
    WHERE HRLYEARN > 0 AND HRLYEARN < 500
      AND LFSSTAT IN (1, 2)
      AND AGE BETWEEN 25 AND 54
""")

# Labels
prov_labels = {
    10: 'Newfoundland & Labrador', 11: 'Prince Edward Island',
    12: 'Nova Scotia', 13: 'New Brunswick', 24: 'Quebec',
    35: 'Ontario', 46: 'Manitoba', 47: 'Saskatchewan',
    48: 'Alberta', 59: 'British Columbia'
}

educ_labels = {
    0: 'Less than high school', 1: 'High school graduate',
    2: 'Some post-secondary', 3: 'Post-secondary certificate',
    4: "Bachelor's degree", 5: 'Above bachelor\'s'
}

noc_labels = {
    0: 'Management', 1: 'Business/Finance', 2: 'Natural Sciences',
    3: 'Health', 4: 'Social Science/Education', 5: 'Art/Culture',
    6: 'Sales/Service', 7: 'Trades', 8: 'Resources/Agriculture',
    9: 'Manufacturing'
}

df['PROV_LABEL'] = df['PROV'].map(prov_labels)
df['EDUC_LABEL'] = df['EDUC'].map(educ_labels)
df['NOC_LABEL'] = df['NOC_10'].map(noc_labels)
df['LOG_HRLYEARN'] = np.log(df['HRLYEARN'])

print(f"\n📊 Sample: N = {len(df):,}")
print(f"   Years: {df['YEAR'].min()}-{df['YEAR'].max()}")
print(f"   Female: {df['IS_FEMALE'].mean():.1%}")

---
## Table 1: Descriptive Statistics

In [ ]:
# ============================================================================
# TABLE 1: DESCRIPTIVE STATISTICS
# ============================================================================

def weighted_stats(data, weight_col):
    """Calculate weighted mean and std."""
    weights = data[weight_col]
    return {
        'mean': np.average(data['HRLYEARN'], weights=weights),
        'std': np.sqrt(np.average((data['HRLYEARN'] - np.average(data['HRLYEARN'], weights=weights))**2, weights=weights)),
        'n': len(data)
    }

# Overall statistics
table1_data = []

# By gender
for gender, label in [(0, 'Men'), (1, 'Women')]:
    subset = df[df['IS_FEMALE'] == gender]
    stats_dict = weighted_stats(subset, 'FINALWT')
    table1_data.append({
        'Category': 'Gender',
        'Group': label,
        'Mean Wage': stats_dict['mean'],
        'Std Dev': stats_dict['std'],
        'N': stats_dict['n']
    })

# By education
for educ in sorted(df['EDUC'].dropna().unique()):
    if educ not in educ_labels:
        continue
    subset = df[df['EDUC'] == educ]
    if len(subset) < 100:
        continue
    stats_dict = weighted_stats(subset, 'FINALWT')
    table1_data.append({
        'Category': 'Education',
        'Group': educ_labels[educ],
        'Mean Wage': stats_dict['mean'],
        'Std Dev': stats_dict['std'],
        'N': stats_dict['n']
    })

# By province (largest 4)
for prov in [35, 24, 59, 48]:  # ON, QC, BC, AB
    subset = df[df['PROV'] == prov]
    stats_dict = weighted_stats(subset, 'FINALWT')
    table1_data.append({
        'Category': 'Province',
        'Group': prov_labels[prov],
        'Mean Wage': stats_dict['mean'],
        'Std Dev': stats_dict['std'],
        'N': stats_dict['n']
    })

table1 = pd.DataFrame(table1_data)
table1['Mean Wage'] = table1['Mean Wage'].round(2)
table1['Std Dev'] = table1['Std Dev'].round(2)

print("=" * 70)
print("TABLE 1: DESCRIPTIVE STATISTICS")
print("=" * 70)
print(table1.to_string(index=False))

In [ ]:
# ============================================================================
# EXPORT TABLE 1 (LaTeX FORMAT)
# ============================================================================

latex_table1 = r"""
\begin{table}[htbp]
\centering
\caption{Descriptive Statistics: Hourly Wages by Demographic Characteristics}
\label{tab:descriptive}
\begin{tabular}{llccc}
\toprule
Category & Group & Mean (\$) & Std Dev & N \\
\midrule
"""

current_category = None
for _, row in table1.iterrows():
    if row['Category'] != current_category:
        if current_category is not None:
            latex_table1 += r"\midrule" + "\n"
        current_category = row['Category']
    
    latex_table1 += f"& {row['Group']} & {row['Mean Wage']:.2f} & {row['Std Dev']:.2f} & {row['N']:,} \\\\\n"

latex_table1 += r"""
\bottomrule
\end{tabular}
\begin{tablenotes}
\small
\item Notes: Sample restricted to prime-age workers (25-54) employed at the time of survey.
All statistics are weighted using survey weights (FINALWT).
\item Source: Statistics Canada Labour Force Survey PUMF, 2010-2025.
\end{tablenotes}
\end{table}
"""

with open(output_dir / 'tables' / 'table1_descriptive.tex', 'w') as f:
    f.write(latex_table1)

table1.to_csv(output_dir / 'tables' / 'table1_descriptive.csv', index=False)
print("💾 Table 1 saved (LaTeX and CSV)")

---
## Table 2: Raw Gender Wage Gap

In [ ]:
# ============================================================================
# TABLE 2: RAW GENDER WAGE GAP BY YEAR
# ============================================================================

table2_data = []

for year in sorted(df['YEAR'].unique()):
    subset = df[df['YEAR'] == year]
    
    male = subset[subset['IS_FEMALE'] == 0]
    female = subset[subset['IS_FEMALE'] == 1]
    
    male_wage = np.average(male['HRLYEARN'], weights=male['FINALWT'])
    female_wage = np.average(female['HRLYEARN'], weights=female['FINALWT'])
    gap = (male_wage - female_wage) / male_wage * 100  # As percentage
    
    table2_data.append({
        'Year': year,
        'Male Wage ($)': male_wage,
        'Female Wage ($)': female_wage,
        'Raw Gap (%)': gap,
        'N': len(subset)
    })

table2 = pd.DataFrame(table2_data)

print("=" * 70)
print("TABLE 2: RAW GENDER WAGE GAP BY YEAR")
print("=" * 70)
print(table2.to_string(index=False, float_format='%.2f'))

# Calculate overall change
first_year_gap = table2.iloc[0]['Raw Gap (%)']
last_year_gap = table2.iloc[-1]['Raw Gap (%)']
change = first_year_gap - last_year_gap

print(f"\n--- Summary ---")
print(f"   Gap in {table2.iloc[0]['Year']}: {first_year_gap:.1f}%")
print(f"   Gap in {table2.iloc[-1]['Year']}: {last_year_gap:.1f}%")
print(f"   Change: {change:+.1f} percentage points")

In [ ]:
# ============================================================================
# EXPORT TABLE 2 (LaTeX FORMAT)
# ============================================================================

latex_table2 = r"""
\begin{table}[htbp]
\centering
\caption{Raw Gender Wage Gap by Year, 2010-2025}
\label{tab:raw_gap}
\begin{tabular}{ccccc}
\toprule
Year & Male Wage (\$) & Female Wage (\$) & Raw Gap (\%) & N \\
\midrule
"""

for _, row in table2.iterrows():
    latex_table2 += f"{int(row['Year'])} & {row['Male Wage ($)']:.2f} & {row['Female Wage ($)']:.2f} & {row['Raw Gap (%)']:.1f} & {int(row['N']):,} \\\\\n"

latex_table2 += r"""
\bottomrule
\end{tabular}
\begin{tablenotes}
\small
\item Notes: Raw gap calculated as $(W_M - W_F) / W_M \times 100$.
All wages are weighted using survey weights.
\item Source: Statistics Canada Labour Force Survey PUMF.
\end{tablenotes}
\end{table}
"""

with open(output_dir / 'tables' / 'table2_raw_gap.tex', 'w') as f:
    f.write(latex_table2)

table2.to_csv(output_dir / 'tables' / 'table2_raw_gap.csv', index=False)
print("💾 Table 2 saved (LaTeX and CSV)")

---
## Table 3: Adjusted Gender Wage Gap (Regression)

In [ ]:
# ============================================================================
# TABLE 3: REGRESSION ANALYSIS
# ============================================================================

# Prepare data
reg_df = df.dropna(subset=['LOG_HRLYEARN', 'AGE', 'EDUC', 'PROV']).copy()

# Create dummy variables
reg_df['AGE_SQ'] = reg_df['AGE'] ** 2

# Model specifications
models = []

# Model 1: Unadjusted
X1 = sm.add_constant(reg_df[['IS_FEMALE']])
model1 = sm.WLS(reg_df['LOG_HRLYEARN'], X1, weights=reg_df['FINALWT']).fit()
models.append(('(1) Unadjusted', model1))

# Model 2: Age controls
X2 = sm.add_constant(reg_df[['IS_FEMALE', 'AGE', 'AGE_SQ']])
model2 = sm.WLS(reg_df['LOG_HRLYEARN'], X2, weights=reg_df['FINALWT']).fit()
models.append(('(2) + Age', model2))

# Model 3: + Education
educ_dummies = pd.get_dummies(reg_df['EDUC'], prefix='EDUC', drop_first=True)
X3 = pd.concat([sm.add_constant(reg_df[['IS_FEMALE', 'AGE', 'AGE_SQ']]), educ_dummies], axis=1)
model3 = sm.WLS(reg_df['LOG_HRLYEARN'], X3, weights=reg_df['FINALWT']).fit()
models.append(('(3) + Education', model3))

# Model 4: + Province
prov_dummies = pd.get_dummies(reg_df['PROV'], prefix='PROV', drop_first=True)
X4 = pd.concat([X3, prov_dummies], axis=1)
model4 = sm.WLS(reg_df['LOG_HRLYEARN'], X4, weights=reg_df['FINALWT']).fit()
models.append(('(4) + Province', model4))

# Model 5: + Occupation
noc_dummies = pd.get_dummies(reg_df['NOC_10'], prefix='NOC', drop_first=True)
X5 = pd.concat([X4, noc_dummies], axis=1)
model5 = sm.WLS(reg_df['LOG_HRLYEARN'], X5, weights=reg_df['FINALWT']).fit()
models.append(('(5) + Occupation', model5))

# Extract results
print("=" * 70)
print("TABLE 3: GENDER WAGE GAP - REGRESSION ANALYSIS")
print("Dependent Variable: Log(Hourly Earnings)")
print("=" * 70)

print(f"\n{'Model':<20} {'Female Coef':<12} {'Std Err':<10} {'Gap %':<10} {'R²':<8}")
print("-" * 60)

table3_data = []
for name, model in models:
    coef = model.params['IS_FEMALE']
    se = model.bse['IS_FEMALE']
    gap_pct = (np.exp(coef) - 1) * 100
    r2 = model.rsquared
    
    # Significance stars
    pval = model.pvalues['IS_FEMALE']
    stars = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    
    print(f"{name:<20} {coef:>8.4f}{stars:<3} ({se:.4f})  {gap_pct:>6.1f}%     {r2:.4f}")
    
    table3_data.append({
        'Model': name,
        'Coefficient': coef,
        'Std Error': se,
        'P-value': pval,
        'Gap (%)': gap_pct,
        'R-squared': r2,
        'N': int(model.nobs)
    })

print(f"\nN = {int(model5.nobs):,}")
print("\nNote: *** p<0.001, ** p<0.01, * p<0.05")

In [ ]:
# ============================================================================
# EXPORT TABLE 3 (LaTeX - Publication Style)
# ============================================================================

# Prepare table DataFrame
table3_df = pd.DataFrame(table3_data)

# Create LaTeX header (start only; details added in later cells)
latex_table3 = r"""
\begin{table}[htbp]
\centering
\caption{Gender Wage Gap: Regression Analysis}
\label{tab:regression}
\begin{tabular}{lccccc}
\toprule
 & (1) & (2) & (3) & (4) & (5) \\
 & Unadjusted & + Age & + Educ & + Prov & + Occ \\
\midrule
Female & """

In [ ]:
# Add coefficients and standard errors (formatted for LaTeX)
coefs = []
ses = []
for _, row in table3_df.iterrows():
    pval = row['P-value']
    stars = '^{***}' if pval < 0.001 else '^{**}' if pval < 0.01 else '^{*}' if pval < 0.05 else ''
    coefs.append(f"${row['Coefficient']:.4f}{stars}$")
    ses.append(f"({row['Std Error']:.4f})")

latex_table3 += " & ".join(coefs) + " \\\\\n"
latex_table3 += " & " + " & ".join(ses) + " \\\\\n"
latex_table3 += r"\midrule" + "\n"

In [ ]:
# Add gap percentages, R-squared, controls and finalize LaTeX
# Add gap percentage
gaps = [f"{row['Gap (%)']:.1f}\\%" for _, row in table3_df.iterrows()]
latex_table3 += "Implied Gap (\\%) & " + " & ".join(gaps) + " \\\\\n"

# Add R-squared
r2s = [f"{row['R-squared']:.3f}" for _, row in table3_df.iterrows()]
latex_table3 += "R$^2$ & " + " & ".join(r2s) + " \\\\\n"

# Add controls indicators
latex_table3 += r"""
\midrule
Age controls & No & Yes & Yes & Yes & Yes \\
Education controls & No & No & Yes & Yes & Yes \\
Province FE & No & No & No & Yes & Yes \\
Occupation FE & No & No & No & No & Yes \\
\bottomrule
\end{tabular}
\begin{tablenotes}
\small
\item Notes: Dependent variable is log hourly earnings. All regressions weighted using survey weights.
Standard errors in parentheses. $^{*}p<0.05$, $^{**}p<0.01$, $^{***}p<0.001$.
\item Source: Statistics Canada LFS PUMF, 2010-2025.
\end{tablenotes}
\end{table}
"""

# Save outputs
with open(output_dir / 'tables' / 'table3_regression.tex', 'w') as f:
    f.write(latex_table3)

table3_df.to_csv(output_dir / 'tables' / 'table3_regression.csv', index=False)
print("💾 Table 3 saved (LaTeX and CSV)")

---
## Figure 1: Gender Wage Gap Trend

In [ ]:
# ============================================================================
# FIGURE 1: GENDER WAGE GAP TREND (PUBLICATION QUALITY)
# ============================================================================

fig, ax = plt.subplots(figsize=(8, 5))

years = table2['Year']
gaps = table2['Raw Gap (%)']

# Main line
ax.plot(years, gaps, 'o-', color='#1f77b4', linewidth=2, markersize=6, 
        label='Raw gender wage gap')

# Add trend line
z = np.polyfit(years, gaps, 1)
p = np.poly1d(z)
ax.plot(years, p(years), '--', color='gray', linewidth=1.5, 
        label=f'Linear trend ({z[0]:.2f}pp/year)')

# COVID marker
ax.axvline(x=2020, color='red', linestyle=':', linewidth=1.5, alpha=0.7)
ax.annotate('COVID-19', xy=(2020, gaps.max()*0.95), fontsize=9, color='red')

ax.set_xlabel('Year')
ax.set_ylabel('Gender Wage Gap (%)')
ax.set_title('Gender Wage Gap in Canada, 2010-2025')
ax.legend(loc='upper right', frameon=True)

ax.set_ylim(0, gaps.max() * 1.1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(output_dir / 'figures' / 'figure1_gap_trend.png', dpi=300, bbox_inches='tight')
plt.savefig(output_dir / 'figures' / 'figure1_gap_trend.pdf', bbox_inches='tight')
plt.show()

print("💾 Figure 1 saved (PNG and PDF)")

---
## Figure 2: Gap by Education

In [ ]:
# ============================================================================
# FIGURE 2: GAP BY EDUCATION
# ============================================================================

# Calculate gaps by education
educ_gaps = []
for educ in sorted(df['EDUC'].dropna().unique()):
    if educ not in educ_labels:
        continue
    subset = df[df['EDUC'] == educ]
    if len(subset) < MIN_GROUP_N:
        continue
    
    male = subset[subset['IS_FEMALE'] == 0]
    female = subset[subset['IS_FEMALE'] == 1]
    
    male_wage = np.average(male['HRLYEARN'], weights=male['FINALWT'])
    female_wage = np.average(female['HRLYEARN'], weights=female['FINALWT'])
    gap = (male_wage - female_wage) / male_wage * 100
    
    educ_gaps.append({
        'Education': educ_labels[educ],
        'Gap': gap,
        'Order': educ
    })

educ_df = pd.DataFrame(educ_gaps).sort_values('Order')

# Plot
fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.barh(educ_df['Education'], educ_df['Gap'], color='steelblue', edgecolor='black')

# Add values on bars
for bar, gap in zip(bars, educ_df['Gap']):
    ax.annotate(f'{gap:.1f}%', 
                xy=(gap + 0.5, bar.get_y() + bar.get_height()/2),
                va='center', fontsize=10)

ax.set_xlabel('Gender Wage Gap (%)')
ax.set_title('Gender Wage Gap by Education Level')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(output_dir / 'figures' / 'figure2_gap_education.png', dpi=300, bbox_inches='tight')
plt.savefig(output_dir / 'figures' / 'figure2_gap_education.pdf', bbox_inches='tight')
plt.show()

print("💾 Figure 2 saved")

---
## Figure 3: Gap by Province

In [ ]:
# ============================================================================
# FIGURE 3: GAP BY PROVINCE (MAP-STYLE)
# ============================================================================

# Calculate gaps by province
prov_gaps = []
for prov in df['PROV'].unique():
    if prov not in prov_labels:
        continue
    subset = df[df['PROV'] == prov]
    if len(subset) < MIN_GROUP_N:
        continue
    
    male = subset[subset['IS_FEMALE'] == 0]
    female = subset[subset['IS_FEMALE'] == 1]
    
    male_wage = np.average(male['HRLYEARN'], weights=male['FINALWT'])
    female_wage = np.average(female['HRLYEARN'], weights=female['FINALWT'])
    gap = (male_wage - female_wage) / male_wage * 100
    
    prov_gaps.append({
        'Province': prov_labels[prov],
        'Gap': gap,
        'N': len(subset)
    })

prov_df = pd.DataFrame(prov_gaps).sort_values('Gap', ascending=True)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

# Color by gap size
colors = plt.cm.RdYlGn_r((prov_df['Gap'] - prov_df['Gap'].min()) / 
                          (prov_df['Gap'].max() - prov_df['Gap'].min()))

bars = ax.barh(prov_df['Province'], prov_df['Gap'], color=colors, edgecolor='black')

# Add values
for bar, gap in zip(bars, prov_df['Gap']):
    ax.annotate(f'{gap:.1f}%', 
                xy=(gap + 0.3, bar.get_y() + bar.get_height()/2),
                va='center', fontsize=9)

# Add national average line
national_gap = (prov_df['Gap'] * prov_df['N']).sum() / prov_df['N'].sum()
ax.axvline(x=national_gap, color='black', linestyle='--', linewidth=2, label=f'National avg: {national_gap:.1f}%')

ax.set_xlabel('Gender Wage Gap (%)')
ax.set_title('Gender Wage Gap by Province')
ax.legend(loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(output_dir / 'figures' / 'figure3_gap_province.png', dpi=300, bbox_inches='tight')
plt.savefig(output_dir / 'figures' / 'figure3_gap_province.pdf', bbox_inches='tight')
plt.show()

print("💾 Figure 3 saved")

---
## Figure 4: Regression Coefficients

In [ ]:
# ============================================================================
# FIGURE 4: COEFFICIENT PLOT
# ============================================================================

fig, ax = plt.subplots(figsize=(8, 5))

# Data from regression table
model_names = table3_df['Model'].str.replace(r'\(\d+\) ', '', regex=True)
gaps = table3_df['Gap (%)'].values

# Calculate confidence intervals
ci_lower = []
ci_upper = []
for _, model in models:
    coef = model.params['IS_FEMALE']
    se = model.bse['IS_FEMALE']
    ci_lower.append((np.exp(coef - 1.96*se) - 1) * 100)
    ci_upper.append((np.exp(coef + 1.96*se) - 1) * 100)

# Plot with error bars
y_pos = range(len(model_names))
ax.errorbar(gaps, y_pos, 
            xerr=[np.array(gaps) - np.array(ci_lower), 
                  np.array(ci_upper) - np.array(gaps)],
            fmt='o', capsize=5, capthick=2, markersize=10, color='steelblue')

ax.set_yticks(y_pos)
ax.set_yticklabels(model_names)
ax.set_xlabel('Implied Gender Wage Gap (%)')
ax.set_title('Gender Wage Gap: Effect of Adding Controls')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

# Add value labels
for i, (g, up) in enumerate(zip(gaps, ci_upper)):
    ax.annotate(f'{g:.1f}%', xy=(up + 0.5, i), va='center', fontsize=9)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(output_dir / 'figures' / 'figure4_coefficients.png', dpi=300, bbox_inches='tight')
plt.savefig(output_dir / 'figures' / 'figure4_coefficients.pdf', bbox_inches='tight')
plt.show()

print("💾 Figure 4 saved")

---
## Summary Statistics Export

In [ ]:
# ============================================================================
# EXPORT SUMMARY STATISTICS (JSON)
# ============================================================================

# Collect key statistics
summary_stats = {
    'metadata': {
        'generated': datetime.now().isoformat(),
        'data_source': 'Statistics Canada LFS PUMF',
        'years': f"{df['YEAR'].min()}-{df['YEAR'].max()}",
        'sample_size': int(len(df)),
        'sample_restriction': 'Prime-age workers (25-54), employed'
    },
    'raw_gap': {
        'latest_year': int(table2.iloc[-1]['Year']),
        'latest_gap_pct': float(table2.iloc[-1]['Raw Gap (%)']),
        'earliest_gap_pct': float(table2.iloc[0]['Raw Gap (%)']),
        'change_pp': float(table2.iloc[0]['Raw Gap (%)'] - table2.iloc[-1]['Raw Gap (%)'])
    },
    'adjusted_gap': {
        'unadjusted_pct': float(table3_df.iloc[0]['Gap (%)']),
        'fully_adjusted_pct': float(table3_df.iloc[-1]['Gap (%)']),
        'explained_share': float((table3_df.iloc[0]['Gap (%)'] - table3_df.iloc[-1]['Gap (%)']) / table3_df.iloc[0]['Gap (%)'] * 100)
    },
    'by_education': educ_df.set_index('Education')['Gap'].to_dict(),
    'by_province': prov_df.set_index('Province')['Gap'].to_dict()
}

with open(output_dir / 'summary_statistics.json', 'w') as f:
    json.dump(summary_stats, f, indent=2)

print("=" * 70)
print("PUBLICATION OUTPUTS - SUMMARY")
print("=" * 70)
print(f"\n📁 Output directory: {output_dir}")
print(f"\n📊 Tables generated:")
for f in (output_dir / 'tables').glob('*'):
    print(f"   - {f.name}")

print(f"\n📈 Figures generated:")
for f in (output_dir / 'figures').glob('*'):
    print(f"   - {f.name}")

print(f"\n📋 Summary statistics: summary_statistics.json")

print("\n" + "=" * 70)
print("KEY FINDINGS")
print("=" * 70)
print(f"\n1. Raw gender wage gap ({summary_stats['raw_gap']['latest_year']}): {summary_stats['raw_gap']['latest_gap_pct']:.1f}%")
print(f"2. Gap change ({table2.iloc[0]['Year']:.0f}-{summary_stats['raw_gap']['latest_year']}): {summary_stats['raw_gap']['change_pp']:+.1f}pp")
print(f"3. Adjusted gap (full controls): {summary_stats['adjusted_gap']['fully_adjusted_pct']:.1f}%")
print(f"4. Share explained by observables: {summary_stats['adjusted_gap']['explained_share']:.0f}%")

print("\n✅ All publication outputs generated successfully!")

---

## Data Availability Statement

The data used in this analysis are from the Statistics Canada Labour Force Survey Public Use Microdata Files (LFS PUMF), available through the Statistics Canada Data Liberation Initiative.

## Replication Code

All code is available at: https://github.com/equipay-canada/gender-wage-gap

## Citation

```bibtex
@article{equipay2025,
  title={Gender Wage Gap in Canada: A Comprehensive Analysis},
  author={EquiPay Canada Research Team},
  year={2025},
  note={Data from Statistics Canada LFS PUMF 2010-2025}
}
```